In [1]:
import nltk
from nltk import ne_chunk, pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize

for resource, location in {
    "punkt_tab": "tokenizers/punkt_tab/english",
    "stopwords": "corpora/stopwords",
    "wordnet": "corpora/wordnet.zip",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
    "maxent_ne_chunker_tab": "chunkers/maxent_ne_chunker_tab",
    "words": "corpora/words"
}.items():
    '''
    nltk.data.find(location) 會先檢查這些資料是否已存在；
    如果沒有，就透過 nltk.download(resource) 自動下載。
    '''
    try:
        nltk.data.find(location)
    except LookupError:
        if not nltk.download(resource, quiet=True):
            raise RuntimeError(f"無法取得 NLTK 資料：{resource}")

In [7]:
text = "Alice visited London yesterday. She enjoyed reading books in the library."

sentences = sent_tokenize(text)
sentence_tokens = [word_tokenize(sentence) for sentence in sentences]
sentence_tokens

[['Alice', 'visited', 'London', 'yesterday', '.'],
 ['She', 'enjoyed', 'reading', 'books', 'in', 'the', 'library', '.']]

In [9]:
tagged_sentences = [pos_tag(tokens) for tokens in sentence_tokens]
tagged_sentences

[[('Alice', 'NNP'),
  ('visited', 'VBD'),
  ('London', 'NNP'),
  ('yesterday', 'NN'),
  ('.', '.')],
 [('She', 'PRP'),
  ('enjoyed', 'VBD'),
  ('reading', 'VBG'),
  ('books', 'NNS'),
  ('in', 'IN'),
  ('the', 'DT'),
  ('library', 'NN'),
  ('.', '.')]]

In [13]:
s = "Alice"
s[0]

'A'

In [10]:
for tagged in tagged_sentences:
    print(ne_chunk(tagged))

(S Alice/NNP visited/VBD (GPE London/NNP) yesterday/NN ./.)
(S
  She/PRP
  enjoyed/VBD
  reading/VBG
  books/NNS
  in/IN
  the/DT
  library/NN
  ./.)


In [20]:
lemmatizer = WordNetLemmatizer()
pos_map = {"J": wordnet.ADJ, "V": wordnet.VERB,
           "N": wordnet.NOUN, "R": wordnet.ADV}

lemmatizer.lemmatize("play", pos=wordnet.VERB)

stop_words = set(stopwords.words("english"))
clean_tokens = []

for tagged in tagged_sentences:
    for word, tag in tagged:
        lower = word.lower()

        if not lower.isalpha() or lower in stop_words:
            continue

        pos = pos_map.get(tag[0])

        lemma = lemmatizer.lemmatize(lower, pos=pos) if pos else lower

        clean_tokens.append(lemma)

clean_tokens

['alice', 'visit', 'london', 'yesterday', 'enjoy', 'read', 'book', 'library']

In [21]:
import spacy
import pandas as pd
from spacy import displacy
from IPython.display import HTML, display

nlp = spacy.load("en_core_web_sm")
doc = nlp("The students read a book in the library. They discussed the story with their teacher.")

In [22]:
def dependency_table(doc):
    """將逐詞分析整理成便於閱讀的表格。"""
    return pd.DataFrame([
        {"i": token.i, "text": token.text, "pos": token.pos_,
         "tag": token.tag_, "dep": token.dep_,
         "head_i": token.head.i, "head": token.head.text,
         "說明": spacy.explain(token.dep_) or "請參考模型標籤定義"}
        for token in doc if not token.is_space
    ])

dependency_table(doc)

,i,text,pos,tag,dep,head_i,head,說明
0,0,The,DET,DT,det,1,students,determiner
1,1,students,NOUN,NNS,nsubj,2,read,nominal subject
2,2,read,VERB,VBD,ROOT,2,read,root
3,3,a,DET,DT,det,4,book,determiner
4,4,book,NOUN,NN,dobj,2,read,direct object
5,5,in,ADP,IN,prep,2,read,prepositional modifier
6,6,the,DET,DT,det,7,library,determiner
7,7,library,NOUN,NN,pobj,5,in,object of preposition
8,8,.,PUNCT,.,punct,2,read,punctuation
9,9,They,PRON,PRP,nsubj,10,discussed,nominal subject


In [27]:
def show_dependencies(doc):
    for sentence in doc.sents:
        print(sentence.text)
        svg = displacy.render(
            sentence,
            style = "dep",
            options={
                "compact": False, # 顯示完整箭頭
                "distance": 110, # 箭頭長度
                "font": "Microsoft JhengHei, Arial, sans-serif", 
            }
        )

        display(HTML(f'<div style="overflow-x:auto; padding:16px">{svg}</div>'))

show_dependencies(doc)

The students read a book in the library.


They discussed the story with their teacher.
